# 02 · Laboratorio de estudio del bastón O–I–T

## 00 · La pregunta central
¿Cómo puede un evento de luz cambiar el potencial de una terminal distante sin
producir un spike? Estudiaremos una sola célula: tres regiones eléctricas conectadas.
La entrada será una variación de corriente; evolucionarán tres voltajes; la salida
observable será el potencial de T, **no** una cantidad de glutamato.

La ruta es **intuición fisiológica → concepto físico → formulación → deducción →
predicción → experimento → interpretación**. Necesitas EDO y álgebra lineal; la
fisiología se introduce aquí. Escribe las predicciones antes de ejecutar cada bloque.
Las respuestas plegadas tienen una alternativa en las preguntas de estudio.

En cada afirmación distinguimos: **[Dato experimental]**, **[Hipótesis de modelado]**,
**[Consecuencia matemática]** y **[Resultado numérico]**. No basta que una curva sea
convincente para convertirla en medición. Todas las figuras pueden exportarse a SVG.

Preparación: instala `.[dev,widgets]`, selecciona el kernel del entorno y ejecuta
las celdas en orden. Los widgets son opcionales. Las funciones del laboratorio
siguen disponibles mediante llamadas Python. No se necesitan PDF privados.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from IPython.display import display, Markdown
from neurofisiologia import (RodParameters, PhotonEvents, SimulationResult,
    simulate, dark_equilibrium, decompose_currents, current_balance)
from neurofisiologia.model import conductance_matrix
from neurofisiologia.analysis import (compare_with_reference, response_metrics,
    voltage_deviation, rc_step_response, modal_times_ms, variation_of_constants,
    sensitivity_map, poisson_ensemble)
from neurofisiologia.plotting import (plot_regions, plot_circuit, plot_kernel_events,
    plot_voltages, plot_currents, plot_topologies, plot_sensitivity,
    plot_poisson_variability, plot_comparison)

root = next((directory for directory in (Path.cwd(), *Path.cwd().parents)
             if (directory / "src/neurofisiologia").is_dir()
             and (directory / "pyproject.toml").is_file()), None)
if root is None:
    raise RuntimeError("Abre el notebook dentro del repositorio instalado.")
output_dir = root / "outputs/estudio_baston"
output_dir.mkdir(parents=True, exist_ok=True)
parameters = RodParameters()  # Valores iniciales ilustrativos y no calibrados.
time_ms = np.linspace(0, 16000, 801)
single_event = PhotonEvents.from_times([1000])


def show_figure(figure: Figure, name: str) -> None:
    """Muestra y exporta SVG/PNG locales; no añade resultados a Git."""
    with plt.rc_context({"svg.fonttype": "none"}):
        figure.savefig(output_dir / (name+".svg"))
        figure.savefig(output_dir / (name+".png"), dpi=150)
    plt.show()
    plt.close(figure)


def show_metrics(result: SimulationResult) -> None:
    """Lee mínimos muestreados (mV), instantes/retardos (ms) y transmisión."""
    measures = response_metrics(result)
    rows = ["| Región | min ΔV (mV) | t mínimo (ms) |",
            "| --- | ---: | ---: |"]
    for j, region in enumerate("OIT"):
        moment = measures["time_min_ms"][j]
        time_label = "sin respuesta resuelta" if not np.isfinite(moment) else f"{moment:.1f}"
        rows.append(f"| {region} | {measures['min_delta_mv'][j]:.4f} | {time_label} |")
    display(Markdown("\n".join(rows)))
    print("Retardo I respecto de O (ms):", measures["delay_i_ms"])
    print("Retardo T respecto de O (ms):", measures["delay_t_ms"])
    print("Razón |min ΔV_T|/|min ΔV_O|:", measures["transmission_ratio"])

## 01 · Un bastón real y sus regiones
Un bastón (*rod photoreceptor*) es un fotorreceptor retinal especializado en
responder a iluminación tenue. O contiene los discos con pigmento; I sostiene
el metabolismo y los gradientes; T participa en la comunicación sináptica real.
Aquí I también agrupa el cuerpo y la conexión hacia T: tres nodos no reconstruyen
cada detalle anatómico. El dibujo es conceptual, **no a escala**.

[Hipótesis de modelado] Cada región tiene un voltaje uniforme. O–I e I–T son
caminos axiales dentro de la misma célula, no sinapsis ni conexiones entre neuronas.
C y g no se dedujeron de las dimensiones del dibujo. En el bastón real T usa
mecanismos sinápticos; en este modelo solo calculamos V_T.

Fuente anatómica: [Webvision, Photoreceptors, fig. 5 y apartado Light microscopy](https://www.ncbi.nlm.nih.gov/books/NBK11522/).
No se transfieren cifras de primates al bastón de sapo de Baylor.

In [ ]:
figure, axis = plot_regions()
show_figure(figure, "anatomia_conceptual")

## 02 · Por qué la luz reduce una corriente entrante
En oscuridad, canales de la membrana externa permiten entrada de cationes.
La luz activa pigmento y una cadena de señalización: **fotón → rodopsina →
transducina → PDE → cGMP disminuye → canales CNG se cierran → disminuye corriente
oscura entrante**. PDE es una enzima; cGMP, un mensajero; CNG significa canal
activado por nucleótidos cíclicos. Esta secuencia es fisiología de contexto,
**no variables adicionales del programa**.

Menos entrada positiva, mientras persisten otras corrientes, favorece que el
interior sea más negativo: hiperpolarización. Un fotón no se trata como una carga
que entra por un cable. Su absorción desencadena una respuesta eléctrica efectiva.

[Dato experimental] La reducción de corriente oscura fue observada en bastones
([Hagins et al., 1970; retina de rata, resumen](https://pubmed.ncbi.nlm.nih.gov/5439318/)).
La conductancia fotosensible y sensible a cGMP se estudió directamente en
[Yau y Nakatani, 1985](https://doi.org/10.1038/317252a0).
La referencia cuantitativa de este laboratorio sigue siendo **sapo**, Baylor.

[Hipótesis de modelado] Sustituimos la cadena por una variación impuesta negativa
I_photo(t). No añadimos cierre dinámico de canales, ni saturación, ni adaptación.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Reducir una corriente entrante no implica invertir toda la corriente oscura. I_photo es el cambio respecto del fondo. El balance neto se vuelve transitoriamente negativo y disminuye V. La cascada real explica la motivación; no se resuelve aquí.</p></details>

## 03 · Vocabulario, referencia exterior y signos
$V=V_{interior}-V_{exterior}$. Con exterior como referencia 0 mV, V=-40 mV dice
que el interior está 40 mV por debajo. No dice que exista una corriente de -40.
Corriente es flujo de carga por tiempo (pA); capacitancia C mide carga almacenada
por diferencia de potencial (pF); conductancia g relaciona corriente y diferencia
impulsora de potencial (nS).

El potencial de reversión E es aquel donde la corriente de una rama vale cero.
El reposo es donde **la suma de corrientes** vale cero. Solo coinciden bajo
condiciones particulares. Una fuga lineal es una descripción agregada; no se
identifica automáticamente con un canal de potasio.

**Pregunta:** con E=-40 mV y V=-45 mV, ¿la fuga entrante $-g_L(V-E)$ es positiva
 o negativa? **Predicción:** escribe su signo y qué intenta hacer con V.

Coherencia dimensional: $\mathrm{nS\,mV=pA}$ y $\mathrm{pF\,mV/ms=pA}$.
Son magnitudes totales por nodo, no densidades por área.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>V−E=−5 mV. La fuga entrante vale +5 gL pA y empuja V hacia E. Con V mayor que E ocurre lo contrario. No confundas fuga restauradora con reposo acoplado.</p></details>

## 04 · Un compartimento: deducir antes de integrar
[Hipótesis] C es constante. La diferencia de carga relevante se representa por
$Q=CV$; derivar da $\dot Q=C\dot V$. Conservación de carga exige que $\dot Q$
sea la suma de corrientes entrantes. Si hay fuga y una entrada impuesta:

$$C\dot V=-g_L(V-E_L)+I(t).$$

En oscuridad $I=0$, poner $\dot V=0$ da $V^*=E_L$. Restar E y dividir por C
produce $\dot u=-(g_L/C)u$, por tanto $u(t)=u(0)e^{-t/\tau}$ con $\tau=C/g_L$.
Para corriente constante I0 desde t0 e inicio V(t0)=E, el equilibrio desplazado
es $E+I_0/g_L$. Resolver la EDO homogénea alrededor de ese equilibrio da:

$$V(t)=E+\frac{I_0}{g_L}\bigl(1-e^{-(t-t_0)/\tau}\bigr),\quad t\ge t_0.$$

[Consecuencia matemática] A una τ se alcanza 1−exp(−1)≈63.2% del desplazamiento.
No es una respuesta fotónica: la entrada constante permanece encendida.

**Experimento RC didáctico.** ¿Duplicar C duplica la amplitud final o el tiempo?
Predice antes de ejecutar. Se conservan g=0.05 nS, E=-40 mV e I0=-0.3 pA;
solo varía C. Las curvas son evaluaciones de la fórmula analítica, no solve_ivp.

In [ ]:
figure, axis = plt.subplots(figsize=(9, 4), layout="constrained")
for capacitance in (20, 40):
    voltage = rc_step_response(time_ms, capacitance_pf=capacitance, current_pa=-.3)
    axis.plot(time_ms, voltage, label=f"C={capacitance} pF; τ={capacitance/.05:g} ms")
axis.set(xlabel="Tiempo (ms)", ylabel="V analítico (mV)",
         title="Entrada CONSTANTE didáctica desde 1000 ms; E=-40 mV, gL=0.05 nS, I0=-0.3 pA")
axis.legend()
axis.grid(alpha=.2)
show_figure(figure, "rc_analitico")

**Observación:** el límite es -46 mV para ambas C. **Interpretación:** I0/g fija
el desplazamiento estacionario; C/g fija el transitorio. Para una corriente que
termina, una C mayor puede cambiar también el máximo observado.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Duplicar C duplica τ, pero no cambia E+I0/g. No traslades la independencia de amplitud de esta entrada constante a un fotón de duración finita.</p></details>

## 05 · De uno a dos: compartir carga
Conectar O con I permite una corriente proporcional a su diferencia de potencial.
Definimos $J_{OI}=g_{OI}(V_O-V_I)$ positiva de O hacia I. Sale de O y entra a I:

$$C_O\dot V_O=-g_{L,O}(V_O-E_O)-J_{OI}+I_{photo},$$
$$C_I\dot V_I=-g_{L,I}(V_I-E_I)+J_{OI}.$$

[Consecuencia matemática] Los términos −J y +J se cancelan al sumar carga.
Si O se hiperpolariza, J_OI suele ser negativa: I pierde carga positiva hacia O.
O recibe una corriente axial restauradora; I puede hiperpolarizarse.

**Predicción:** bajo el mismo evento, ¿responde I con g_OI=0? ¿Se conserva el
máximo de O al conectarlo? Compara g_OI=0 con 0.10 nS; g_IT=0 en ambos casos.
C, fuga, E y evento se conservan. T está aislada; no se elimina otra EDO.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for coupling, style in ((0, ":"), (.1, "-")):
    pair_parameters = replace(parameters, g_oi_ns=coupling, g_it_ns=0)
    pair = simulate(time_ms, pair_parameters, single_event)
    for j, axis in enumerate(axes):
        axis.plot(time_ms, voltage_deviation(pair)[j], style,
                  label=f"gOI={coupling:g} nS; gIT=0")
        axis.set(title=f"{'OI'[j]}: C={pair_parameters.capacitance_pf[j]} pF; gL={pair_parameters.leak_ns[j]} nS; E=-40 mV",
                 xlabel="Tiempo (ms)", ylabel="ΔV simulado (mV)")
        axis.legend()
        axis.grid(alpha=.2)
figure.suptitle("Parámetros ilustrativos | Misma entrada impuesta: A=1.2 pA, pico=2200 ms, evento en 1000 ms")
show_figure(figure, "pareja_OI")

**Interpretación:** I permanece en reposo aislada. La corriente que la hace
responder procede del vecino, no de una fotocorriente adicional. El efecto sobre
la amplitud de O depende de la carga eléctrica conectada; no es una conservación
de amplitud, sino de corrientes axiales iguales y opuestas.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Con gOI=0, I no tiene entrada distinta de su fuga y comienza en reposo. Al conectarla, el intercambio axial modifica ambas derivadas. Comprueba los signos usando los dos voltajes de la gráfica.</p></details>

## 06 · De dos a tres: dar a T una función eléctrica
Añadimos $J_{IT}=g_{IT}(V_I-V_T)$ positiva I→T. I ahora recibe J_OI y pierde
J_IT; T recibe J_IT. Los balances completos, sin omitir ningún término, son:

$$C_O\dot V_O=-g_{L,O}(V_O-E_O)+g_{OI}(V_I-V_O)+I_{photo}(t),$$
$$C_I\dot V_I=-g_{L,I}(V_I-E_I)+g_{OI}(V_O-V_I)+g_{IT}(V_T-V_I),$$
$$C_T\dot V_T=-g_{L,T}(V_T-E_T)+g_{IT}(V_I-V_T).$$

Sumarlos da $\sum C_j\dot V_j=-\sum g_{L,j}(V_j-E_j)+I_{photo}$:
los axiales se cancelan. No se conserva la carga total de los nodos aislada del
exterior, porque fugas e input intercambian corriente con él.

**Propósito de F1:** identificar qué almacena, qué disipa, qué conecta y dónde
entra la variación de corriente. Las flechas esquemáticas indican la orientación
positiva de referencia, no la dirección instantánea. C, gL y E son ilustrativos.

In [ ]:
figure, axis = plot_circuit(parameters)
show_figure(figure, "F1_circuito")

<details><summary>Respuesta razonada: abre después de predecir</summary><p>gIT=0 impide corriente desde I hacia T. C_T controla su rapidez y gL_T su restauración. T sigue teniendo una EDO, pero sin conexión ni entrada parte de su propio reposo. No hay glutamato calculado.</p></details>

## 07 · El reposo oscuro no es una lista de reversión
En oscuridad, anular cada derivada produce tres ecuaciones algebraicas:

$$(g_{L,O}+g_{OI})V_O-g_{OI}V_I=g_{L,O}E_O,$$
$$-g_{OI}V_O+(g_{L,I}+g_{OI}+g_{IT})V_I-g_{IT}V_T=g_{L,I}E_I,$$
$$-g_{IT}V_I+(g_{L,T}+g_{IT})V_T=g_{L,T}E_T.$$

Ahora sí podemos reunir **los balances ya deducidos** en $K V_{dark}=G_L E$.
K tiene conductancias nS, $G_L=diag(g_L)$, V y E tienen mV. K es invertible
con fugas positivas; más adelante probaremos esa propiedad.

**Predicción:** si E=(-45,-40,-35) mV, ¿puedes inicializar V en E y esperar reposo?
**Experimento:** cambiar solo E, resolver el sistema y comprobar las corrientes.

In [ ]:
unequal_parameters = replace(parameters, reversal_mv=(-45, -40, -35))
unequal_rest = dark_equilibrium(unequal_parameters)
wrong_balance = current_balance(0, unequal_parameters.reversal_mv, unequal_parameters, PhotonEvents())
correct_balance = current_balance(0, unequal_rest, unequal_parameters, PhotonEvents())
print("E (mV):", unequal_parameters.reversal_mv)
print("V_dark acoplado (mV):", unequal_rest)
print("Corriente neta en V=E (pA):", wrong_balance)
print("Corriente neta en reposo calculado (pA):", correct_balance)
np.testing.assert_allclose(correct_balance, 0, atol=1e-12)

**Observación e interpretación:** en V=E las fugas son cero, pero hay diferencias
entre vecinos y las corrientes axiales no son cero. En V_dark las fugas compensan
las corrientes axiales. Con todos los E=-40 mV, V_dark=-40 en todos los nodos.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Potencial de reversión anula una rama; reposo anula la suma. Al haber E distintos, los nodos pueden sostener corrientes opuestas aun con derivadas nulas.</p></details>

## 08 · Un evento efectivo y el experimento de Baylor
[Dato experimental] Baylor, Lamb y Yau (1979) registraron corriente de segmento
externo de bastones de **sapo** con electrodo de succión. Para célula 1, la tabla
1 (p. 621; fig. 3, p. 618) da A≈1.2 pA; fig. 4, p. 619, da pico≈2.2 s.
No midieron aquí nuestros tres voltajes ni calibraron nuestras C o g.
[Artículo público](https://doi.org/10.1113/jphysiol.1979.sp012716).

[Hipótesis] Aproximamos la forma con $h(u)=(u/t_p)^3 e^{3(1-u/t_p)}$ para u≥0,
y cero antes, $t_p=2200$ ms. [Consecuencia] $d\log h/du=3/u-3/t_p$ se anula en
t_p y cambia de positivo a negativo: el máximo es h=1. Por eso −A h tiene
mínimo −A en el pico. El factor cúbico es fenomenológico, sin identificar
cuatro reacciones moleculares. Variar A es un experimento numérico, no afirmar
que cada fotón real produce mayor amplitud si sube el brillo.

**Predicción:** ¿coinciden el tiempo del evento y el pico de corriente?
**F2** los separa y muestra entrada impuesta; no reproduce una traza experimental.

In [ ]:
single_response = simulate(time_ms, parameters, single_event)
figure, axes = plot_kernel_events(single_response)
show_figure(figure, "F2_kernel_eventos")

<details><summary>Respuesta razonada: abre después de predecir</summary><p>El evento ocurre en 1000 ms y el pico de corriente en 3200 ms. La respuesta tiene una forma temporal continua; el evento no es un spike ni un impulso de carga instantáneo.</p></details>

## 09 · Primera respuesta: potencial absoluto y desviación
**Pregunta:** ¿cómo distinguir el nivel de reposo de la amplitud de respuesta?
**Predicción:** espera ΔV<0, atenuación hacia T y mínimos de voltaje posteriores
al pico de corriente. Estos últimos son hipótesis a confrontar con el experimento,
no números tomados de Baylor.

**F3** conserva los colores O/I/T y marca mínimos muestreados. Las métricas usan
min(ΔV), t_min, retardos respecto de O y |min ΔV_T|/|min ΔV_O|.
Una respuesta no resuelta produce NaN/None para instantes o retrasos; no inventa
un retraso cero. El cociente es cero si T es nula y O responde.

In [ ]:
figure, axes = plot_voltages(single_response)
show_figure(figure, "F3_V_y_delta")
show_metrics(single_response)

**Interpretación:** el máximo de hiperpolarización depende de la corriente previa
y del filtrado eléctrico, no solo de su valor instantáneo. Los retardos de mínimos
son descriptores de formas temporales; no son un tiempo de vuelo ni una velocidad
de conducción. Se miden en la malla de salida: reducir su separación mejora la
localización, y una ventana corta puede cortar el mínimo verdadero.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>El voltaje de reposo es una referencia; ΔV mide el cambio. El pico de corriente no obliga a dV/dt=0 porque fuga y axiales siguen actuando. Un mínimo de V requiere corriente neta nula, no corriente foto nula.</p></details>

## 10 · Las corrientes que la curva de voltaje oculta
**Pregunta:** durante la recuperación, ¿debe haber desaparecido la fotocorriente?
**Predicción:** puede seguir siendo negativa mientras fuga y acoplamiento producen
una corriente neta positiva.

**F4** muestra cada corriente entrante por nodo. En I, ambas conexiones se ven
por separado; en T la fotocorriente es exactamente cero. En **F5** seleccionamos
una muestra temporal, imprimimos fuga, axiales, foto, neta y neta/C y damos sentido
a las flechas. Una flecha se invierte con el signo de corriente; una corriente
cero no lleva flecha. El esquema no es una animación de iones individuales.

In [ ]:
figure, axes = plot_currents(single_response)
show_figure(figure, "F4_corrientes")


def inspect_time(t_ms: float = 3440) -> dict:
    """Inspecciona la muestra más cercana; corrientes pA y derivada mV/ms."""
    index = int(np.argmin(abs(single_response.time_ms-t_ms)))
    selected = float(single_response.time_ms[index])
    components = decompose_currents(selected, single_response.voltage_mv[:, index], single_response.parameters, single_response.stimulus)
    figure, axis = plot_circuit(result=single_response, time_ms=t_ms)
    show_figure(figure, "F5_estado_temporal")
    print("Tiempo solicitado / muestra (ms):", t_ms, selected)
    for key in ("leak_current_pa", "axial_oi_by_node_pa", "axial_it_by_node_pa", "photocurrent_pa", "net_current_pa"):
        print(key, "(O,I,T), pA:", np.round(components[key], 6))
    print("Neta/C (O,I,T), mV/ms:", components["net_current_pa"]/single_response.parameters.capacitance_pf)
    assert abs(components["axial_current_pa"].sum()) < 1e-12
    return components

selected_currents = inspect_time(3440)

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Durante recuperación, la neta puede ser positiva con foto todavía negativa. En el mínimo del voltaje la neta cambia de signo. Verifica la suma y divide por C; no atribuyas el cambio de signo a una sola curva.</p></details>

## 11 · Aislamiento, pareja y trío
**Pregunta:** ¿qué condición impide que la respuesta llegue a T?
**Predicción:** romper O-I o I-T basta. Compara O aislado, O-I con T aislada y
O-I-T. **F6** conserva evento, C, fugas y E; solo cambia conectividad eléctrica.
Los tres nodos siguen en el solver. Acompañamos las curvas con medidas, incluso
cuando una región no responde.

In [ ]:
figure, axes = plot_topologies(time_ms, single_event, parameters)
show_figure(figure, "F6_topologias")
for name, varied in (
    ("O aislado", replace(parameters, g_oi_ns=0, g_it_ns=0)),
    ("Pareja O-I", replace(parameters, g_it_ns=0)),
    ("Trío O-I-T", parameters),
):
    print(name)
    show_metrics(simulate(time_ms, varied, single_event))

<details><summary>Respuesta razonada: abre después de predecir</summary><p>T responde únicamente si ambos caminos axiales permiten intercambio. Su respuesta refleja transmisión pasiva y fugas, no liberación sináptica. La menor respuesta de O al conectar nodos no contradice conservación de carga.</p></details>

## 12 · Parámetros: experimentos de una variable y sensibilidad
**Pregunta:** ¿C, fuga y acoplamiento hacen lo mismo? Predice primero lo que
ocurre al duplicar C_O con todas las conductancias fijas. Después cambia solo
fuga de O o una conexión. La referencia discontinua conserva el escenario inicial.
La siguiente función permite también cambiar los parámetros restantes; sus
valores y unidades se declaran explícitamente, y los demás se conservan.

In [ ]:
membrane_comparison = None


def membrane_experiment(capacitance_o_pf: float = 20, g_oi_ns: float = .1,
                        g_it_ns: float = .1, leak_o_ns: float = .05,
                        capacitance_i_pf: float = 15, capacitance_t_pf: float = 5,
                        leak_i_ns: float = .1, leak_t_ns: float = .05,
                        reversal_o_mv: float = -40, reversal_i_mv: float = -40,
                        reversal_t_mv: float = -40, view: str = "delta") -> SimulationResult:
    """Compara membrana modificada con referencia canónica; misma entrada."""
    global membrane_comparison
    varied = RodParameters(
        capacitance_pf=(capacitance_o_pf, capacitance_i_pf, capacitance_t_pf),
        leak_ns=(leak_o_ns, leak_i_ns, leak_t_ns),
        reversal_mv=(reversal_o_mv, reversal_i_mv, reversal_t_mv),
        g_oi_ns=g_oi_ns, g_it_ns=g_it_ns,
    )
    reference, modified = compare_with_reference(time_ms, modified_parameters=varied)
    membrane_comparison = (reference, modified)
    figure, axes = plot_comparison(reference, modified, voltage_view=view,
        modified_condition="Solo membrana; entrada canónica fija")
    show_figure(figure, "comparacion_membrana")
    print("Cambian solo los parámetros eléctricos que elegiste; tiempos y A se conservan.")
    show_metrics(modified)
    return modified

membrane_result = membrane_experiment(capacitance_o_pf=40)

**Interpretación:** una C mayor almacena más carga para cambiar un mismo voltaje,
pero aquí la corriente no es constante; máximo y retraso dependen de la forma.
Cambiar fuga altera restauración y carga efectiva. Cambiar acoplamiento altera
simultáneamente distribución y modos del sistema; no equivale a cambiar solo C.

**F7 pregunta:** ¿qué amplitud llega a T al variar ambas conexiones? Se conservan
C, fugas, E, evento y malla. Cada celda calcula |min ΔV_T| en mV; la escala es
lineal y el dominio mostrado es exactamente el evaluado. No es una identificación
paramétrica ni una sensibilidad infinitesimal, sino un barrido numérico.

In [ ]:
conductances_ns = np.array([0, .05, .1, .15, .2, .3])
sensitivity_mv = sensitivity_map(conductances_ns, conductances_ns, time_ms, single_event)
figure, axis = plot_sensitivity(conductances_ns, conductances_ns, sensitivity_mv)
show_figure(figure, "F7_sensibilidad")

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Los bordes con gOI=0 o gIT=0 no transmiten a T. No concluyas monotonía universal solo por los colores: la conexión también carga nodos previos y cambia la dinámica. C/gL de un nodo aislado deja de describir todo el circuito.</p></details>

## 13 · Muchos eventos: linealidad y azar no son lo mismo
**Pregunta:** ¿dos eventos coincidentes producen el doble de respuesta? Predice
qué se suma: la corriente y la desviación, no el potencial absoluto.
[Consecuencia matemática] El sistema es afín en V y lineal en u=V−V_dark.
Por eso sumar entradas, con misma membrana y reposo inicial, suma u.
Esto se deriva de las ecuaciones; no valida superposición fisiológica bajo luz intensa.

In [ ]:
coincident = simulate(time_ms, parameters, PhotonEvents.from_times([1000, 1000]))
superposition_error_mv = np.max(abs(voltage_deviation(coincident)-2*voltage_deviation(single_response)))
print("Error numérico de superposición (mV):", superposition_error_mv)
assert superposition_error_mv < 2e-6

Cambiar tasa de eventos efectivos no es aumentar A por evento. **F8** muestra
los trenes y corrientes de una realización, y la media con dispersión entre
realizaciones en ΔV_T. Semillas explícitas hacen reproducible el cálculo; no
eliminan la variabilidad. La banda ±1 DE muestral no es intervalo de confianza
ni dato experimental. Las mismas semillas entre tasas no aseguran los mismos
tiempos: nuestro algoritmo no utiliza adelgazamiento compartido.

**Predicción:** ¿debe cada realización a tasa alta hiperpolarizar más que cualquiera
a tasa baja? Contrasta esa afirmación con curvas individuales y estadísticos.
Una banda media±DE puede cruzar cero aunque las realizaciones sean no positivas:
no es un rango físico de todas las trayectorias.

In [ ]:
ensemble_seeds = tuple(range(2026, 2038))
ensembles = {rate: poisson_ensemble(time_ms, rate, ensemble_seeds) for rate in (.3, .6)}
figure, axes = plot_poisson_variability(ensembles, ensemble_seeds)
show_figure(figure, "F8_poisson")
for rate, responses in ensembles.items():
    event_counts = [len(response.stimulus.times_ms) for response in responses]
    print(rate, "eventos/s; media de N:", np.mean(event_counts),
          "; DE de N:", np.std(event_counts, ddof=1), "; semillas:", ensemble_seeds)

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Una tasa es un parámetro de distribución, no el conteo de una muestra. Pueden darse realizaciones vacías o inversiones del orden individual. La linealidad implica escalamiento de la respuesta media con la tasa bajo hipótesis fijas, no orden de todas las muestras.</p></details>

### Controles de entrada, membrana y tiempo
Las funciones anteriores son la alternativa sin widgets. El explorador de entrada
separa A (pA), pico (ms), tasa (eventos/s) y semilla. Su referencia permanece fija.
Al variar solo A conserva los eventos; al variar tasa muestra las dos entradas.
Los controles de membrana empiezan por C_O y conexiones; los demás se agrupan
como avanzados. Cambia una variable por vez. El control temporal usa la muestra
más cercana y muestra la suma de corrientes. Todos requieren un botón de ejecución.
La vista de membrana permite elegir V o ΔV. Si cambias E, ΔV resta el reposo propio
de cada escenario para separar cambio de fondo de amplitud de respuesta.

In [ ]:
input_comparison = None


def input_experiment(amplitude_pa: float = 1.2, t_peak_ms: float = 2200,
                     rate_hz: float = .3, seed: int = 2026) -> SimulationResult:
    """Modifica solo entrada; conserva una referencia fija y muestra dos estímulos."""
    global input_comparison
    reference, modified = compare_with_reference(time_ms, amplitude_pa=amplitude_pa,
        t_peak_ms=t_peak_ms, rate_hz=rate_hz, seed=seed)
    input_comparison = (reference, modified)
    figure, axes = plot_comparison(reference, modified,
        modified_condition=f"A={amplitude_pa:g} pA; pico={t_peak_ms:g} ms; tasa={rate_hz:g}/s; semilla={seed}")
    show_figure(figure, "comparacion_entrada")
    print("Modificada: A (pA), pico (ms), tasa (/s), semilla:", amplitude_pa, t_peak_ms, rate_hz, seed)
    print("Eventos referencia / modificada:", reference.stimulus.times_ms, modified.stimulus.times_ms)
    show_metrics(modified)
    return modified

input_result = input_experiment(amplitude_pa=2.4)

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Sin widgets: usa inspect_time, membrane_experiment e input_experiment con argumentos.")
else:
    input_controls = widgets.interactive(input_experiment,
        {"manual": True, "manual_name": "Comparar entrada"},
        amplitude_pa=widgets.FloatSlider(value=1.2, min=.2, max=2.4, step=.2, description="A (pA)"),
        t_peak_ms=widgets.FloatSlider(value=2200, min=500, max=4000, step=100, description="Pico (ms)"),
        rate_hz=widgets.FloatSlider(value=.3, min=0, max=.6, step=.1, description="Eventos/s"),
        seed=widgets.IntText(value=2026, description="Semilla"))
    time_controls = widgets.interactive(inspect_time,
        {"manual": True, "manual_name": "Inspeccionar tiempo"},
        t_ms=widgets.FloatSlider(value=3440, min=0, max=16000, step=20, description="Tiempo (ms)"))
    membrane_controls = widgets.interactive(membrane_experiment,
        {"manual": True, "manual_name": "Comparar membrana"},
        capacitance_o_pf=widgets.FloatSlider(value=20, min=5, max=80, step=5, description="CO (pF)"),
        g_oi_ns=widgets.FloatSlider(value=.1, min=0, max=.3, step=.05, description="gOI (nS)"),
        g_it_ns=widgets.FloatSlider(value=.1, min=0, max=.3, step=.05, description="gIT (nS)"),
        leak_o_ns=widgets.FloatText(value=.05, description="gLO (nS)"),
        capacitance_i_pf=widgets.FloatText(value=15, description="CI (pF)"),
        capacitance_t_pf=widgets.FloatText(value=5, description="CT (pF)"),
        leak_i_ns=widgets.FloatText(value=.1, description="gLI (nS)"),
        leak_t_ns=widgets.FloatText(value=.05, description="gLT (nS)"),
        reversal_o_mv=widgets.FloatText(value=-40, description="EO (mV)"),
        reversal_i_mv=widgets.FloatText(value=-40, description="EI (mV)"),
        reversal_t_mv=widgets.FloatText(value=-40, description="ET (mV)"),
        view=widgets.Dropdown(options=[("Desviación ΔV", "delta"), ("Potencial V", "V")], value="delta", description="Vista"))
    advanced = widgets.Accordion(children=[widgets.VBox(membrane_controls.children[3:-2])])
    advanced.set_title(0, "Membrana: parámetros avanzados (ilustrativos)")
    display(input_controls, time_controls, widgets.VBox([
        widgets.VBox(membrane_controls.children[:3]), advanced,
        *membrane_controls.children[-2:]]))
    reset_button = widgets.Button(description="Reiniciar parámetros")
    control_groups = (input_controls, time_controls, membrane_controls)
    original_values = {id(group): {child.description: child.value for child in group.children if hasattr(child, "value")}
                       for group in control_groups}
    def reset_experiments(button: widgets.Button) -> None:
        """Restaura valores iniciales y recalcula las tres vistas."""
        for group in control_groups:
            for child in group.children:
                if hasattr(child, "value"):
                    child.value = original_values[id(group)][child.description]
            group.update()
    reset_button.on_click(reset_experiments)
    display(reset_button)

## 14 · Estructura matemática y contraste numérico (ampliación)
Con los tres balances comprendidos, escribimos $C=diag(C_O,C_I,C_T)$,
$G_L=diag(g_{L,O},g_{L,I},g_{L,T})$ y

$$K=\begin{pmatrix}g_{L,O}+g_{OI}&-g_{OI}&0\\
-g_{OI}&g_{L,I}+g_{OI}+g_{IT}&-g_{IT}\\
0&-g_{IT}&g_{L,T}+g_{IT}\end{pmatrix}.$$

El sistema es $C\dot V=-KV+G_LE+e_O I(t)$, $e_O=(1,0,0)^T$. Del equilibrio
$KV_{dark}=G_LE$, sustituir $V=u+V_{dark}$ cancela los términos constantes:
$C\dot u=-Ku+e_O I(t)$. No se ha añadido ninguna física.

Para cualquier x distinto de cero:
$$x^TKx=\sum_j g_{L,j}x_j^2+g_{OI}(x_O-x_I)^2+g_{IT}(x_I-x_T)^2>0.$$
Con fugas positivas, K es definida positiva. $C^{-1}K$ no es generalmente
simétrica, pero es similar a $C^{-1/2}KC^{-1/2}$; tiene autovalores reales positivos.
Los modos decaen, con $\tau_m=1/\lambda_m$ ms. En oscuridad la energía de desviación
$W=\tfrac12u^TCu$ cumple $\dot W=-u^TKu<0$ fuera del reposo.
El sistema pasivo no ofrece excitabilidad regenerativa ni oscilaciones autónomas.

Con $M=-C^{-1}K$, variación de constantes da la identidad exacta del modelo:
$$u(t)=e^{M(t-t_0)}u(t_0)+\int_{t_0}^t e^{M(t-s)}C^{-1}e_O I(s)\,ds.$$
La curva calculada con expm y cuadratura también es **numérica**; la identidad,
no la evaluación flotante, es exacta. Se contrasta con solve_ivp sin reemplazarlo.

**Predicción:** ¿es suficiente tomar C_O/gL_O como tiempo del trío?
**Experimento:** compara constantes modales y dos formas independientes de cálculo.

In [ ]:
print("K, nS:", conductance_matrix(parameters))
print("C/gL aislados, ms:", np.array(parameters.capacitance_pf)/parameters.leak_ns)
print("Constantes modales acopladas, ms:", modal_times_ms(parameters))
validation_time_ms = np.linspace(0, 12000, 25)
quadrature_voltage = variation_of_constants(validation_time_ms, parameters, single_event)
ivp_voltage = simulate(validation_time_ms, parameters, single_event).voltage_mv
independent_error_mv = np.max(abs(quadrature_voltage-ivp_voltage))
print("Diferencia solve_ivp frente a expm+cuadratura (mV):", independent_error_mv)
assert independent_error_mv < 2e-7
refined = simulate(time_ms, parameters, single_event, max_step_ms=5)
print("Diferencia al reducir paso máximo a 5 ms (mV):",
      np.max(abs(refined.voltage_mv-single_response.voltage_mv)))

<details><summary>Respuesta razonada: abre después de predecir</summary><p>C/gL es exacta para un nodo aislado. El trío tiene modos colectivos dependientes de todas las C y g. La positividad prueba estabilidad del modelo, no la ausencia de otros mecanismos en el bastón real.</p></details>

## 15 · Alcance y autoevaluación

**Dato:** escala de corriente de sapo en Baylor. **Hipótesis:** discretización,
C/g/E ilustrativos y corriente impuesta. **Consecuencia:** conservación axial,
linealidad en ΔV y estabilidad. **Resultado numérico:** amplitudes, retardos y
sensibilidad de las figuras. Ninguna categoría sustituye otra.

No podemos inferir voltajes reales calibrados, eficiencia cuántica, saturación,
adaptación, concentración de cGMP, liberación de glutamato ni spikes. Un bastón
real tiene mecanismos activos: que nuestro sistema sea pasivo no los niega.
Quedan para otro estudio, junto con conos y amacrinas.

Antes de mirar respuestas, explica: (1) por qué I_photo es negativa, (2) por qué
reposo y reversión difieren, (3) por qué las corrientes axiales se cancelan, (4) qué
significa una respuesta nula, (5) qué comparación cambia solo A y cuál introduce
azar, y (6) qué afirmaciones no se pueden contrastar con la corriente de Baylor.

Para estudiar sin ejecutar: [notas y deducciones](../docs/estudio_baston_modelo_minimo.md),
[glosario bilingüe](../docs/glosario_neurofisio.md) y
[preguntas con respuestas](../docs/preguntas_de_estudio_baston.md).
[Fuentes y versiones](../docs/referencias_y_parametros.md) diferencia el localizador
36 del documento de estudio de septiembre, reportado en la decisión aprobada,
del 38 verificado en la versión pública de agosto de Herrera-Valdez.
Los PDF privados no se abrieron para esta implementación.

Las figuras SVG/PNG se guardan en `outputs/estudio_baston/`; ambos notebooks
fuente permanecen sin salidas. Las pruebas y CI ejecutan el modelo reutilizable,
los notebooks, los controles y la alternativa sin widgets.

<details><summary>Respuesta razonada: abre después de predecir</summary><p>Respuestas: menos corriente entrante produce neta negativa; reposo anula suma, reversión anula rama; axiales transfieren dentro de una célula; respuesta nula carece de tiempo al mínimo útil; A conserva eventos y tasa cambia realizaciones; una corriente medida no calibra tres voltajes ni sinapsis.</p></details>